In [11]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import seaborn as sns

In [ ]:

all_data = []

for page in range(1, 100): 
    url = f"https://www.bayut.eg/en/egypt/properties-for-sale/page-{page}/"
    
    res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(res.text, "html.parser")

    script = soup.find("script", {"type": "application/ld+json"})
    
    if not script:
        print(f"Page {page} skipped")
        continue

    data = json.loads(script.string)
    items = data["@graph"][0]["mainEntity"]["itemListElement"]

    for item in items:
        prop = item["item"]["mainEntity"]
        seller = prop.get("seller", {})
        agency = seller.get("memberOf", {}).get("name")

        all_data.append({
            "price": prop.get("price"),
            "bedrooms": prop.get("numberOfBedrooms"),
            "bathrooms": prop.get("numberOfBathroomsTotal"),
            "area": prop.get("floorSize", {}).get("value"),
            "property_type": prop.get("accommodationCategory"),
            "city": prop.get("address", {}).get("addressLocality"),
            "region": prop.get("address", {}).get("addressRegion"),
            "latitude": prop.get("geo", {}).get("latitude"),
            "longitude": prop.get("geo", {}).get("longitude"),
            "agency": agency
        })

    print(f"Page {page} done")

df = pd.DataFrame(all_data)



In [12]:
print(df.shape)

(2338, 11)


In [16]:
df.head(5)

,price,bedrooms,bathrooms,area,property_type,city,region,latitude,longitude,agency,price_per_m
0,15708100,4,4,220,Penthouse,Katameya,Cairo,29.980463,31.385464,Property Hills,71400
1,3000000,2,2,108,Chalet,Ain Sukhna,Suez,29.392860,32.539915,Golden Leaves Real Estate,27777
2,5000000,4,4,327,Apartment,6th of October,Giza,29.970731,31.036953,Property Hills,15290
3,6300000,3,3,167,Apartment,New Cairo,Cairo,30.071321,31.577261,Realm Properties,37724
4,11800000,3,3,530,Townhouse,Sheikh Zayed,Giza,30.038144,31.038177,Property Hills,22264


In [14]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
Index: 2338 entries, 0 to 2351
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   price          2338 non-null   int64  
 1   bedrooms       2338 non-null   int64  
 2   bathrooms      2338 non-null   Int64  
 3   area           2338 non-null   int64  
 4   property_type  2338 non-null   str    
 5   city           2338 non-null   str    
 6   region         2338 non-null   str    
 7   latitude       2338 non-null   float64
 8   longitude      2338 non-null   float64
 9   agency         2338 non-null   str    
 10  price_per_m    2338 non-null   int64  
dtypes: Int64(1), float64(2), int64(4), str(4)
memory usage: 221.5 KB


,price,bedrooms,bathrooms,area,latitude,longitude,price_per_m
count,2.338000e+03,2338.000000,2338.0,2338.000000,2338.000000,2338.000000,2.338000e+03
mean,1.464727e+07,3.129170,3.007271,214.503422,29.928565,31.262557,7.421115e+04
std,1.822360e+07,1.366353,1.427596,151.088517,0.876786,1.168552,3.326142e+05
min,3.800000e+05,0.000000,1.0,2.000000,26.859520,27.237316,7.000000e+03
25%,6.264000e+06,2.000000,2.0,130.000000,29.989265,30.991040,3.888200e+04
50%,9.800000e+06,3.000000,3.0,173.500000,30.046842,31.483846,5.739900e+04
75%,1.680000e+07,4.000000,4.0,240.000000,30.087053,31.600855,8.079350e+04
max,3.500000e+08,11.000000,11.0,1500.000000,31.490098,33.971972,1.600000e+07


In [17]:
df["bathrooms"] = df["bathrooms"].astype("Int64")

In [ ]:
df["area"] = pd.to_numeric(df["area"].str.replace(r"\D", "", regex=True), errors="coerce")

In [ ]:
df = df.dropna(subset=["bathrooms","agency"])

In [ ]:
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = Q1 - 1.5 * IQR

In [ ]:
df["price_per_m"] = (df["price"] / df["area"]).astype(int)

In [ ]:
df.sort_values(by="price_per_m", ascending=False).head(10)